# SCF Phase 2 shard 13

Sweeps: crossover, nullcal. Jobs: 24. Projected: 4.7 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 500, \"p\": 1000, \"r\": 3, \"l\": [0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"825be77cf3de\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 4000, \"raw_path\": \"data/sim/nullcal/raw/825be77cf3de.parquet\", \"means_path\": \"data/sim/nullcal/means/825be77cf3de.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 3, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"f0290738339e\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 4000, \"raw_path\": \"data/sim/nullcal/raw/f0290738339e.parquet\", \"means_path\": \"data/sim/nullcal/means/f0290738339e.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"b13d705199ef\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 4000, \"raw_path\": \"data/sim/nullcal/raw/b13d705199ef.parquet\", \"means_path\": \"data/sim/nullcal/means/b13d705199ef.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"3a7bf5f1f69c\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 4000, \"raw_path\": \"data/sim/nullcal/raw/3a7bf5f1f69c.parquet\", \"means_path\": \"data/sim/nullcal/means/3a7bf5f1f69c.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 4000, \"p\": 800, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"be6f508e92e0\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 400, \"raw_path\": \"data/sim/nullcal/raw/be6f508e92e0.parquet\", \"means_path\": \"data/sim/nullcal/means/be6f508e92e0.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"5b336518841e\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 1200, \"raw_path\": \"data/sim/nullcal/raw/5b336518841e.parquet\", \"means_path\": \"data/sim/nullcal/means/5b336518841e.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"697fc3d5e64d\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 1200, \"raw_path\": \"data/sim/nullcal/raw/697fc3d5e64d.parquet\", \"means_path\": \"data/sim/nullcal/means/697fc3d5e64d.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 3, \"l\": [0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"cc4b5f3877b3\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 1200, \"raw_path\": \"data/sim/nullcal/raw/cc4b5f3877b3.parquet\", \"means_path\": \"data/sim/nullcal/means/cc4b5f3877b3.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 3, \"l\": [0.9486832980505138, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"8f99551fef33\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 1200, \"raw_path\": \"data/sim/nullcal/raw/8f99551fef33.parquet\", \"means_path\": \"data/sim/nullcal/means/8f99551fef33.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"5ce56d26a290\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 500, \"raw_path\": \"data/sim/nullcal/raw/5ce56d26a290.parquet\", \"means_path\": \"data/sim/nullcal/means/5ce56d26a290.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"7860f836f61e\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 500, \"raw_path\": \"data/sim/nullcal/raw/7860f836f61e.parquet\", \"means_path\": \"data/sim/nullcal/means/7860f836f61e.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 1.3416407864998738, 1.3416407864998738], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"ce498755b791\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 500, \"raw_path\": \"data/sim/nullcal/raw/ce498755b791.parquet\", \"means_path\": \"data/sim/nullcal/means/ce498755b791.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 1.3416407864998738, 1.3416407864998738], \"theta\": 1.5707963267948966, \"profile\": \"super\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"30ca44277dc8\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 500, \"raw_path\": \"data/sim/nullcal/raw/30ca44277dc8.parquet\", \"means_path\": \"data/sim/nullcal/means/30ca44277dc8.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 4000, \"r\": 3, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"g\": 4.0, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"705b46091a3b\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/705b46091a3b.parquet\", \"means_path\": \"data/sim/crossover/means/705b46091a3b.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"g\": 0.25, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6dca75004866\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/6dca75004866.parquet\", \"means_path\": \"data/sim/crossover/means/6dca75004866.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"g\": 0.5, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"44b1cfde2306\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/44b1cfde2306.parquet\", \"means_path\": \"data/sim/crossover/means/44b1cfde2306.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"g\": 1.0, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"7fa3e1e22f79\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/7fa3e1e22f79.parquet\", \"means_path\": \"data/sim/crossover/means/7fa3e1e22f79.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"g\": 2.0, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"69d69e6c5347\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/69d69e6c5347.parquet\", \"means_path\": \"data/sim/crossover/means/69d69e6c5347.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"g\": 4.0, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"8eaff95bb8a1\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/8eaff95bb8a1.parquet\", \"means_path\": \"data/sim/crossover/means/8eaff95bb8a1.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"g\": 0.25, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"7ffc14e94529\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/7ffc14e94529.parquet\", \"means_path\": \"data/sim/crossover/means/7ffc14e94529.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"g\": 0.5, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"e3ce1da8f7ed\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/e3ce1da8f7ed.parquet\", \"means_path\": \"data/sim/crossover/means/e3ce1da8f7ed.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"g\": 1.0, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"07eeae1b443c\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/07eeae1b443c.parquet\", \"means_path\": \"data/sim/crossover/means/07eeae1b443c.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"g\": 2.0, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"e53b27ab3355\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/e53b27ab3355.parquet\", \"means_path\": \"data/sim/crossover/means/e53b27ab3355.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"g\": 4.0, \"profile\": \"mixed\", \"label\": \"crossover\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"7eda31b0267a\", \"mode\": \"crossover\", \"sweep\": \"crossover\", \"reps\": 150, \"raw_path\": \"data/sim/crossover/raw/7eda31b0267a.parquet\", \"means_path\": \"data/sim/crossover/means/7eda31b0267a.npz\", \"_rank\": 3, \"_sweep\": \"crossover\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 13, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(13), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)